### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="phiusil_phishing",
    dataset_year="2023",
    domain_str="technology & internet",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://archive.ics.uci.edu/dataset/967/phiusiil+phishing+url+dataset",
    download_description="""

    mkdir -p local-data-warehouse/phiusil_phishing 
    wget -P local-data-warehouse/phiusil_phishing/ https://archive.ics.uci.edu/static/public/967/phiusiil+phishing+url+dataset.zip
    unzip local-data-warehouse/phiusil_phishing/phiusiil+phishing+url+dataset.zip -d local-data-warehouse/phiusil_phishing/ 
    rm local-data-warehouse/phiusil_phishing/phiusiil+phishing+url+dataset.zip
""",
    # References
    academic_reference_bibtex="""@article{prasad2024phiusiil,
  title={PhiUSIIL: A diverse security profile empowered phishing URL detection framework based on similarity index and incremental learning},
  author={Prasad, Arvind and Chandra, Shalini},
  journal={Computers \\& Security},
  volume={136},
  pages={103545},
  year={2024},
  publisher={Elsevier}
}

""",
    academic_reference_bibtex_key="prasad2024phiusiil",
    license="CC BY 4.0",
    data_tags=["IID", "ForcedIIDFromTemporal"],
    curation_comments="""
    - We remove some duplicated URLs as they only differ in the URL length by one character, likely stemming from a data collection error.
    - We drop FILE_NAME as it is an identifier
    - 93.5% of the domains are unique. of the non-unique domains, only 0.0026 non-unique domains are phishing websites. Because domains like google.docs are so frequent and simple to classify, we never allow them in the test data and define a custom split.
    - We keep all domains that appear more than once as training data and do a random split for the remaining data. 
    - We don't use the domain column, to focus on generalization based on the provided features instead of memorization of domains.
    - The focus on unseen domains in the test data shifts the task slightly, but also mitigates the fact that we don't have temporal information although this is a temporal task.
    - We rename the labels to "Legitimate" and "Phishing" for better interpretability.
    """,
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="label",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="label",
    # group_on="Domain",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "PhiUSIIL_Phishing_URL_Dataset.csv")

# Drop duplicated URLs that only differ in the URL length by one character, likely stemming from a data collection error.
df = df.loc[df.URL.drop_duplicates().index]
df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)

predefined_train_indices = df.index[df.Domain.map(df.Domain.value_counts()>1)].tolist()

# Drop redundant features
df = df.drop(columns=["FILENAME", "URL", "Domain"])

df.label = df.label.map({0: "Phishing", 1: "Legitimate"})

print("Loaded data shape:", df.shape)

Loaded data shape: (235370, 53)


In [3]:
# print(f"{df.Domain.nunique()/df.shape[0]:.4f} domains are unique")
# print(f"Only {df.loc[df.Domain.map(df.Domain.value_counts()>1),"label"].mean():.4f} non-unique domains are phishing websites")

In [4]:
# # Domains that appear more than once and have fraudulent URLs
# vc = df.Domain.value_counts()
# for domain in vc[vc>1].index:
#     mean_label = df.loc[df.Domain==domain,"label"].mean()
#     if mean_label > 0:
#         print(domain, mean_label)


In [5]:
# Use if needed to get see all cols of pandas dataframes
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
df.head()

,URLLength,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,URLCharProb,TLDLength,NoOfSubDomain,HasObfuscation,NoOfObfuscatedChar,ObfuscationRatio,NoOfLettersInURL,LetterRatioInURL,NoOfDegitsInURL,DegitRatioInURL,NoOfEqualsInURL,NoOfQMarkInURL,NoOfAmpersandInURL,NoOfOtherSpecialCharsInURL,SpacialCharRatioInURL,IsHTTPS,LineOfCode,LargestLineLength,HasTitle,Title,DomainTitleMatchScore,URLTitleMatchScore,HasFavicon,Robots,IsResponsive,NoOfURLRedirect,NoOfSelfRedirect,HasDescription,NoOfPopup,NoOfiFrame,HasExternalFormSubmit,HasSocialNet,HasSubmitButton,HasHiddenFields,HasPasswordField,Bank,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,38,32,0,host,32.362773,0.565217,0.000045,0.039013,4,3,0,0,0.0,16,0.421,8,0.211,0,0,0,3,0.079,0,117,429,1,s107008tde20fornex,0.0,0.0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,21,1,1,0,0,26,Phishing
1,26,19,0,nl,100.000000,0.750000,0.008200,0.057060,2,1,0,0,0.0,12,0.462,1,0.038,0,0,0,1,0.038,1,2584,9381,1,espresso4you,100.0,100.0,0,1,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,69,21,82,389,1,391,Legitimate
2,22,15,0,com,100.000000,1.000000,0.522907,0.061062,3,1,0,0,0.0,9,0.409,0,0.000,0,0,0,1,0.045,1,1006,12566,1,mazzios,100.0,100.0,0,1,1,0,0,1,0,2,0,1,0,0,0,0,0,0,1,24,14,19,48,0,61,Legitimate
3,27,21,0,com,77.239523,1.000000,0.522907,0.052884,3,1,0,0,0.0,15,0.556,0,0.000,0,0,0,1,0.037,0,2,37,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Phishing
4,29,23,0,com,59.571429,1.000000,0.522907,0.064046,3,1,0,0,0.0,17,0.586,0,0.000,0,0,0,1,0.034,0,2,39,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Phishing


## Data Checks

In [6]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 235,370
Columns: 53
Use sampling: False (sample size: 235,370)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['URLCharProb', 'Title', 'URLSimilarityIndex', 'LargestLineLength', 'LineOfCode', 'NoOfSelfRef', 'NoOfExternalRef', 'NoOfImage', 'CharContinuationRate', 'LetterRatioInURL']
Rows remaining as candidates after top-10 filter: 202 (of 235,370)

#### Duplicate Report
Total duplicate rows: 120 (0.05% of dataset)
Duplicate rows ignoring target: 120 (0.05% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [7]:
# Sample Rows
df_head

,URLLength,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,URLCharProb,TLDLength,NoOfSubDomain,HasObfuscation,NoOfObfuscatedChar,ObfuscationRatio,NoOfLettersInURL,LetterRatioInURL,NoOfDegitsInURL,DegitRatioInURL,NoOfEqualsInURL,NoOfQMarkInURL,NoOfAmpersandInURL,NoOfOtherSpecialCharsInURL,SpacialCharRatioInURL,IsHTTPS,LineOfCode,LargestLineLength,HasTitle,Title,DomainTitleMatchScore,URLTitleMatchScore,HasFavicon,Robots,IsResponsive,NoOfURLRedirect,NoOfSelfRedirect,HasDescription,NoOfPopup,NoOfiFrame,HasExternalFormSubmit,HasSocialNet,HasSubmitButton,HasHiddenFields,HasPasswordField,Bank,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,38,32,0,host,32.362773,0.565217,0.000045,0.039013,4,3,0,0,0.0,16,0.421,8,0.211,0,0,0,3,0.079,0,117,429,1,s107008tde20fornex,0.0,0.0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,21,1,1,0,0,26,Phishing
1,26,19,0,nl,100.000000,0.750000,0.008200,0.057060,2,1,0,0,0.0,12,0.462,1,0.038,0,0,0,1,0.038,1,2584,9381,1,espresso4you,100.0,100.0,0,1,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,69,21,82,389,1,391,Legitimate
2,22,15,0,com,100.000000,1.000000,0.522907,0.061062,3,1,0,0,0.0,9,0.409,0,0.000,0,0,0,1,0.045,1,1006,12566,1,mazzios,100.0,100.0,0,1,1,0,0,1,0,2,0,1,0,0,0,0,0,0,1,24,14,19,48,0,61,Legitimate
3,27,21,0,com,77.239523,1.000000,0.522907,0.052884,3,1,0,0,0.0,15,0.556,0,0.000,0,0,0,1,0.037,0,2,37,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Phishing
4,29,23,0,com,59.571429,1.000000,0.522907,0.064046,3,1,0,0,0.0,17,0.586,0,0.000,0,0,0,1,0.034,0,2,39,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Phishing


In [8]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,URLSimilarityIndex,float64,0.0,0.0,36350.0,"100.0, 66.72, 66.6954, 36.6819, 22.334, 49.3856, 60.8456, 80.0, 9.5408, 23.0309"
1,CharContinuationRate,float64,0.0,0.0,898.0,"1.0, 0.6667, 0.5714, 0.75, 0.6, 0.5, 0.625, 0.8, 0.6364, 0.8333"
2,TLDLegitimateProb,float64,0.0,0.0,465.0,"0.5229, 0.08, 0.0384, 0.0015, 0.0286, 0.006, 0.0129, 0.0327, 0.018, 0.0101"
3,URLCharProb,float64,0.0,0.0,227293.0,"0.0573, 0.0416, 0.0416, 0.0492, 0.0492, 0.0416, 0.0416, 0.0416, 0.0492, 0.0492"
4,ObfuscationRatio,float64,0.0,0.0,146.0,"0.0, 0.056, 0.098, 0.037, 0.03, 0.054, 0.027, 0.032, 0.029, 0.055"
5,LetterRatioInURL,float64,0.0,0.0,709.0,"0.5, 0.48, 0.458, 0.519, 0.435, 0.536, 0.409, 0.552, 0.381, 0.567"
6,DegitRatioInURL,float64,0.0,0.0,575.0,"0.0, 0.176, 0.067, 0.269, 0.045, 0.043, 0.081, 0.038, 0.111, 0.167"
7,SpacialCharRatioInURL,float64,0.0,0.0,240.0,"0.043, 0.042, 0.04, 0.045, 0.038, 0.037, 0.048, 0.036, 0.05, 0.034"
8,DomainTitleMatchScore,float64,0.0,0.0,152.0,"100.0, 0.0, 75.0, 83.3333, 80.0, 66.6667, 87.5, 85.7143, 88.8889, 4.1667"
9,URLTitleMatchScore,float64,0.0,0.0,497.0,"100.0, 0.0, 75.0, 87.5, 83.3333, 80.0, 66.6667, 85.7143, 88.8889, 90.0"


In [9]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
URLLength,235370.0,34.545516,41.332730,13.000000,6.097000e+03
DomainLength,235370.0,21.456022,9.127733,4.000000,1.100000e+02
IsDomainIP,235370.0,0.002702,0.051912,0.000000,1.000000e+00
URLSimilarityIndex,235370.0,78.497537,28.947899,0.155574,1.000000e+02
CharContinuationRate,235370.0,0.845900,0.216384,0.000000,1.000000e+00
TLDLegitimateProb,235370.0,0.260503,0.251618,0.000000,5.229071e-01
URLCharProb,235370.0,0.055760,0.010583,0.001083,9.082366e-02
TLDLength,235370.0,2.764269,0.599667,2.000000,1.300000e+01
NoOfSubDomain,235370.0,1.164957,0.600695,0.000000,1.000000e+01
HasObfuscation,235370.0,0.002052,0.045254,0.000000,1.000000e+00


In [10]:
# Categorical Feature Statistics
cat_stats

value   count    pct
column rank                                            
TLD    1                             com  112382  47.75
       2                             org   18792   7.98
       3                             net    7076   3.01
       4                             app    6467   2.75
       5                              uk    6395   2.72
Title  1                               0   32617  13.86
       2                          #NAME?      21   0.01
       3     65gfgfgfgfg4g4gblogspot?m=1      12   0.01
       4                             gov      11   0.00
       5      info-update-sucreeblogspot      10   0.00
label  1                      Legitimate  134850  57.29
       2                        Phishing  100520  42.71

In [11]:
# Target Distribution
target_df

,count,pct
label,,
Legitimate,134850,57.29
Phishing,100520,42.71


## Task Curation

In [12]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended splits: n_repeats=3, n_splits=3, test_size=None


In [13]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

# -- For IID Data
splits = curation_recommendations.get_recommended_iid_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
)

# Move samples with non-unique domains to the training set and do a random split for the remaining data, ensuring that all samples from the same domain are in the same split.
for rep in splits:
    for fld in splits[rep]:
        train_idx, test_idx = splits[rep][fld]
        test_idx = sorted(list(set(test_idx)-set(predefined_train_indices)))
        splits[rep][fld] = (sorted(set(train_idx).union(predefined_train_indices)), test_idx)
        print(f"Repeat {rep}, Fold {fld}: Train samples: {len(splits[rep][fld][0])}, Test samples: {len(splits[rep][fld][1])}")


splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
)

Using Stratified IID splits.
Repeat 0, Fold 0: Train samples: 163761, Test samples: 71609
Repeat 0, Fold 1: Train samples: 163770, Test samples: 71600
Repeat 0, Fold 2: Train samples: 163744, Test samples: 71626
Repeat 1, Fold 0: Train samples: 163797, Test samples: 71573
Repeat 1, Fold 1: Train samples: 163737, Test samples: 71633
Repeat 1, Fold 2: Train samples: 163741, Test samples: 71629
Repeat 2, Fold 0: Train samples: 163706, Test samples: 71664
Repeat 2, Fold 1: Train samples: 163761, Test samples: 71609
Repeat 2, Fold 2: Train samples: 163808, Test samples: 71562


## Export

In [14]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019d00ac-890c-71bb-8dd5-2f2977a85ba8
f939a3cb76c08edab9316ff8376c04102bf01fe4b6fa64f09966203a507943e5
